# gdmbayes: model variants

How to configure variance structures and spatial effects for `spGDMM` with the
unified string / callable API of `ModelConfig`. This notebook is about
building models, not fitting them — no sampling happens here. See `basic_usage.ipynb`
for fitting and prediction, and `southwest_running_example.ipynb` for a full analysis.

In [1]:

from gdmbayes import ModelConfig, spGDMM

## 1. Built-in variance and spatial effect strings

In [2]:
print("\nVariance options (pass as 'variance' string or callable):")
print("  - 'homogeneous'         : Constant variance across all predictions")
print("  - 'covariate_dependent' : Variance depends on pairwise distance")
print("  - 'polynomial'          : Polynomial variance function of mean")

print("\nSpatial effect options (pass as 'spatial_effect' string or callable):")
print("  - 'none'          : No spatial random effects")
print("  - 'abs_diff'      : Absolute difference in GP spatial effect")
print("  - 'squared_diff'  : Squared difference in GP spatial effect")


Variance options (pass as 'variance' string or callable):
  - 'homogeneous'         : Constant variance across all predictions
  - 'covariate_dependent' : Variance depends on pairwise distance
  - 'polynomial'          : Polynomial variance function of mean

Spatial effect options (pass as 'spatial_effect' string or callable):
  - 'none'          : No spatial random effects
  - 'abs_diff'      : Absolute difference in GP spatial effect
  - 'squared_diff'  : Squared difference in GP spatial effect


## 2. Creating models with string API

In [3]:
model1 = spGDMM(model_config=ModelConfig(variance="homogeneous", spatial_effect="none"))
print(f"\nModel 1: variance={model1._config.variance!r}, spatial_effect={model1._config.spatial_effect!r}")

model2 = spGDMM(model_config=ModelConfig(variance="covariate_dependent", spatial_effect="abs_diff"))
print(f"Model 2: variance={model2._config.variance!r}, spatial_effect={model2._config.spatial_effect!r}")

model3 = spGDMM(model_config=ModelConfig(variance="polynomial", spatial_effect="squared_diff"))
print(f"Model 3: variance={model3._config.variance!r}, spatial_effect={model3._config.spatial_effect!r}")


Model 1: variance='homogeneous', spatial_effect='none'
Model 2: variance='covariate_dependent', spatial_effect='abs_diff'
Model 3: variance='polynomial', spatial_effect='squared_diff'


## 3. Custom variance callable

In [4]:
import pymc as pm


def my_variance(mu, X_sigma):
    """Custom heteroscedastic variance: exponential function of mean."""
    beta_s = pm.HalfNormal("beta_s", sigma=1)
    return beta_s * pm.math.exp(mu)

def my_spatial(psi, row_ind, col_ind):
    """Custom spatial effect: hyperbolic tangent of GP difference."""
    return pm.math.tanh(psi[row_ind] - psi[col_ind])

model_custom = spGDMM(model_config=ModelConfig(variance=my_variance, spatial_effect=my_spatial))
print("\nCustom model:")
print(f"  variance callable: {model_custom._config.variance.__name__}")
print(f"  spatial_effect callable: {model_custom._config.spatial_effect.__name__}")


Custom model:
  variance callable: my_variance
  spatial_effect callable: my_spatial


## 4. to_dict / from_dict round-trip

In [5]:
cfg = ModelConfig(variance="polynomial", spatial_effect="abs_diff")
d = cfg.to_dict()
print(f"\nto_dict(): {d}")

cfg2 = ModelConfig.from_dict(d)
print(f"from_dict(): variance={cfg2.variance!r}, spatial_effect={cfg2.spatial_effect!r}")


to_dict(): {'alpha_importance': True, 'variance': 'polynomial', 'spatial_effect': 'abs_diff'}
from_dict(): variance='polynomial', spatial_effect='abs_diff'


## 5. Invalid strings raise ValueError immediately

In [6]:
try:
    ModelConfig(variance="bad_name")
except ValueError as e:
    print(f"\nModelConfig(variance='bad_name') → ValueError: {e}")

try:
    ModelConfig(spatial_effect="bad_name")
except ValueError as e:
    print(f"ModelConfig(spatial_effect='bad_name') → ValueError: {e}")


ModelConfig(variance='bad_name') → ValueError: Unknown variance='bad_name'. Valid strings: ['covariate_dependent', 'homogeneous', 'polynomial']. Pass a callable for custom variance.
ModelConfig(spatial_effect='bad_name') → ValueError: Unknown spatial_effect='bad_name'. Valid strings: ['abs_diff', 'none', 'squared_diff']. Pass a callable for custom spatial effect.
